# State-Dependent U.S. Equity Sector Rotation
## A Systematic Framework for Trend-Based Active Sector Allocation

# Block 1 — Research Configuration

This block establishes the project environment and freezes the principal research choices before any backtest results are observed.

## Primary comparison

1. S&P 500 passive benchmark  
2. 12-month time-series momentum (TSMOM) sector rotation  
3. State-dependent sector rotation  

## Core constraints

- 11 GICS U.S. equity sectors
- Weekly signal observations
- Signals observed at the **Friday close**
- Target weights calculated after the Friday close
- Portfolio rebalanced on the **next U.S. trading session, normally Monday**
- If Monday is a U.S. market holiday, execution moves to the next available session
- Long-only
- Fully invested
- No leverage
- No outright short positions
- Active sector over/underweights around a strategic allocation
- Parameters specified ex ante rather than optimized immediately

## Canonical timing convention

**Friday close → compute signals and target weights → next U.S. trading session (normally Monday) → rebalance**

A Friday signal must never affect portfolio returns before the next executable U.S. trading session.


In [ ]:
# ============================================================
# BLOCK 1.1 — GOOGLE DRIVE & PROJECT PATHS
# ============================================================

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation"
)

DIRS = {
    "root": PROJECT_ROOT,
    "data_raw": PROJECT_ROOT / "data" / "raw",
    "data_processed": PROJECT_ROOT / "data" / "processed",
    "outputs": PROJECT_ROOT / "outputs",
    "figures": PROJECT_ROOT / "outputs" / "figures",
    "tables": PROJECT_ROOT / "outputs" / "tables",
    "manifests": PROJECT_ROOT / "manifests",
    "logs": PROJECT_ROOT / "logs",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)
print("\nProject directories ready.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root:
/content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation

Project directories ready.


In [ ]:
# ============================================================
# BLOCK 1.2 — IMPORTS & REPRODUCIBILITY
# ============================================================

import json
import math
import platform
import sys
from dataclasses import asdict, dataclass
from datetime import datetime, timezone

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")


Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
NumPy: 2.1.3
pandas: 2.2.3


In [ ]:
# ============================================================
# BLOCK 1.3 — RESEARCH CONFIGURATION
# ============================================================

@dataclass(frozen=True)
class ResearchConfig:

    # --------------------------------------------------------
    # WEEKLY SIGNAL / EXECUTION TIMING
    # --------------------------------------------------------
    # Signals are measured using the completed Friday close.
    signal_frequency: str = "W-FRI"
    signal_observation: str = "FRIDAY_CLOSE"

    # Portfolio rebalancing occurs once per week, but NOT at the
    # same Friday close used to generate the signal.
    rebalance_frequency: str = "WEEKLY"
    rebalance_day: str = "MONDAY"

    # If Monday is a U.S. market holiday, trade on the next
    # available U.S. trading session.
    rebalance_execution_rule: str = "NEXT_US_TRADING_SESSION_AFTER_SIGNAL"

    # A Friday signal must be lagged by at least one executable
    # trading session before it can affect portfolio holdings.
    minimum_signal_execution_lag_sessions: int = 1

    # --------------------------------------------------------
    # PORTFOLIO CONSTRAINTS
    # --------------------------------------------------------
    long_only: bool = True
    fully_invested: bool = True
    leverage_allowed: bool = False
    target_gross_exposure: float = 1.00

    # --------------------------------------------------------
    # STRATEGIC ALLOCATION
    # --------------------------------------------------------
    strategic_weight_method: str = "SP500_SECTOR_WEIGHTS"
    strategic_weight_fallback: str = "INVERSE_VOLATILITY"
    inverse_vol_lookback_weeks: int = 52

    # --------------------------------------------------------
    # ACTIVE STATE-DEPENDENT ALLOCATION
    # --------------------------------------------------------
    active_weight_min: float = -0.50
    active_weight_max: float = 0.50
    active_step: float = 0.10

    # --------------------------------------------------------
    # TSMOM BENCHMARK
    # --------------------------------------------------------
    tsmom_lookback_months: int = 12
    tsmom_active_tilt: float = 0.25

    # --------------------------------------------------------
    # PROPRIETARY INDICATOR PARAMETERS
    # --------------------------------------------------------
    fast_ema: int = 20
    slow_ema: int = 50
    signal_ema: int = 25

    bb_length: int = 20
    bb_smoothing: int = 5
    upper_multiplier: float = 1.0
    lower_multiplier: float = 1.0

    direction_lookback: int = 2

    # --------------------------------------------------------
    # TRADING ASSUMPTIONS
    # --------------------------------------------------------
    transaction_cost_bps: float = 5.0

    # --------------------------------------------------------
    # RESEARCH DISCIPLINE
    # --------------------------------------------------------
    parameter_optimization_enabled: bool = False
    use_lookahead_data: bool = False


CONFIG = ResearchConfig()

CONFIG


ResearchConfig(signal_frequency='W-FRI', signal_observation='FRIDAY_CLOSE', rebalance_frequency='WEEKLY', rebalance_day='MONDAY', rebalance_execution_rule='NEXT_US_TRADING_SESSION_AFTER_SIGNAL', minimum_signal_execution_lag_sessions=1, long_only=True, fully_invested=True, leverage_allowed=False, target_gross_exposure=1.0, strategic_weight_method='SP500_SECTOR_WEIGHTS', strategic_weight_fallback='INVERSE_VOLATILITY', inverse_vol_lookback_weeks=52, active_weight_min=-0.5, active_weight_max=0.5, active_step=0.1, tsmom_lookback_months=12, tsmom_active_tilt=0.25, fast_ema=20, slow_ema=50, signal_ema=25, bb_length=20, bb_smoothing=5, upper_multiplier=1.0, lower_multiplier=1.0, direction_lookback=2, transaction_cost_bps=5.0, parameter_optimization_enabled=False, use_lookahead_data=False)

In [ ]:
# ============================================================
# BLOCK 1.4 — 11-SECTOR UNIVERSE
# ============================================================

SECTOR_UNIVERSE = pd.DataFrame(
    [
        ("Communication Services", "XLC"),
        ("Consumer Discretionary", "XLY"),
        ("Consumer Staples", "XLP"),
        ("Energy", "XLE"),
        ("Financials", "XLF"),
        ("Health Care", "XLV"),
        ("Industrials", "XLI"),
        ("Information Technology", "XLK"),
        ("Materials", "XLB"),
        ("Real Estate", "XLRE"),
        ("Utilities", "XLU"),
    ],
    columns=["sector", "ticker"],
).set_index("ticker")

BENCHMARK_TICKER = "SPY"

assert len(SECTOR_UNIVERSE) == 11
assert SECTOR_UNIVERSE.index.is_unique

display(SECTOR_UNIVERSE)


,sector
ticker,
XLC,Communication Services
XLY,Consumer Discretionary
XLP,Consumer Staples
XLE,Energy
XLF,Financials
XLV,Health Care
XLI,Industrials
XLK,Information Technology
XLB,Materials


## Historical-universe caveat

The present-day 11-sector SPDR mapping is the intended implementation universe, but not every ETF has the same historical coverage.

In particular, XLC and XLRE have shorter histories than the older Select Sector SPDR funds.

Block 2 therefore treats historical coverage as an explicit research problem rather than silently synthesizing unavailable observations. The final sample start date will be determined from defensible point-in-time data availability.


In [ ]:
# ============================================================
# BLOCK 1.5 — STATE-DEFINITION CONSTANTS
# ============================================================

# Underlying / strategic trend state.
STRATEGIC_STATES = [
    "BEARISH",
    "EARLY_REVERSAL",
    "CONFIRMED_TREND",
    "NORMAL_POSITIVE_TREND",
]

# Tactical overlay. This is deliberately kept separate from the
# strategic state to avoid the architecture problem encountered
# in the earlier single-asset implementation.
TACTICAL_STATES = [
    "NONE",
    "PULLBACK_ACCUMULATION",
    "PROFIT_TAKING",
]

VALID_ACTIVE_MULTIPLIER_RANGE = (
    1.0 + CONFIG.active_weight_min,
    1.0 + CONFIG.active_weight_max,
)

print("Strategic states:", STRATEGIC_STATES)
print("Tactical states:", TACTICAL_STATES)
print("Permitted raw-weight multiplier range:", VALID_ACTIVE_MULTIPLIER_RANGE)


Strategic states: ['BEARISH', 'EARLY_REVERSAL', 'CONFIRMED_TREND', 'NORMAL_POSITIVE_TREND']
Tactical states: ['NONE', 'PULLBACK_ACCUMULATION', 'PROFIT_TAKING']
Permitted raw-weight multiplier range: (0.5, 1.5)


In [ ]:
# ============================================================
# BLOCK 1.6 — CONFIGURATION VALIDATION
# ============================================================

def validate_research_config(config: ResearchConfig) -> None:

    # ---------------- Timing ----------------
    assert config.signal_frequency == "W-FRI"
    assert config.signal_observation == "FRIDAY_CLOSE"

    assert config.rebalance_frequency == "WEEKLY"
    assert config.rebalance_day == "MONDAY"

    assert (
        config.rebalance_execution_rule
        == "NEXT_US_TRADING_SESSION_AFTER_SIGNAL"
    )

    assert config.minimum_signal_execution_lag_sessions >= 1

    # ---------------- Portfolio ----------------
    assert config.long_only is True
    assert config.fully_invested is True
    assert config.leverage_allowed is False
    assert config.target_gross_exposure == 1.00

    # ---------------- Active allocation ----------------
    assert -1.0 < config.active_weight_min <= 0.0
    assert config.active_weight_max >= 0.0
    assert config.active_step > 0.0

    # ---------------- TSMOM ----------------
    assert 0.0 <= config.tsmom_active_tilt < 1.0
    assert config.tsmom_lookback_months > 0

    # ---------------- Indicator ----------------
    assert config.fast_ema < config.slow_ema
    assert config.signal_ema > 0

    assert config.bb_length > 1
    assert config.bb_smoothing > 0

    assert config.upper_multiplier > 0
    assert config.lower_multiplier > 0

    assert config.direction_lookback > 0

    # ---------------- Research discipline ----------------
    assert config.transaction_cost_bps >= 0.0
    assert config.parameter_optimization_enabled is False
    assert config.use_lookahead_data is False


validate_research_config(CONFIG)

print("Research configuration validated successfully.")


Research configuration validated successfully.


In [ ]:
# ============================================================
# BLOCK 1.7 — WEEKLY TIMING CONVENTION
# ============================================================

TIMING_CONVENTION = pd.Series(
    {
        "Signal observation": CONFIG.signal_observation,
        "Signal frequency": CONFIG.signal_frequency,
        "Target calculation": "AFTER_FRIDAY_CLOSE",
        "Portfolio rebalance frequency": CONFIG.rebalance_frequency,
        "Nominal rebalance day": CONFIG.rebalance_day,
        "Execution rule": CONFIG.rebalance_execution_rule,
        "Minimum signal-to-execution lag": (
            f"{CONFIG.minimum_signal_execution_lag_sessions} trading session"
        ),
        "Holiday treatment": (
            "If Monday is not a U.S. trading session, "
            "execute on the next available U.S. trading session"
        ),
    },
    name="Weekly timing convention",
).to_frame()

display(TIMING_CONVENTION)

print(
    "\nCanonical sequence:\n"
    "Friday close "
    "-> calculate signals and target weights "
    "-> next U.S. trading session (normally Monday) "
    "-> rebalance"
)


,Weekly timing convention
Signal observation,FRIDAY_CLOSE
Signal frequency,W-FRI
Target calculation,AFTER_FRIDAY_CLOSE
Portfolio rebalance frequency,WEEKLY
Nominal rebalance day,MONDAY
Execution rule,NEXT_US_TRADING_SESSION_AFTER_SIGNAL
Minimum signal-to-execution lag,1 trading session
Holiday treatment,"If Monday is not a U.S. trading session, execu..."



Canonical sequence:
Friday close -> calculate signals and target weights -> next U.S. trading session (normally Monday) -> rebalance


In [ ]:
# ============================================================
# BLOCK 1.8 — SAVE RESEARCH MANIFEST
# ============================================================

manifest = {
    "project": "State-Dependent U.S. Equity Sector Rotation",
    "subtitle": "A Systematic Framework for Trend-Based Active Sector Allocation",
    "block": "Block 1 - Research Configuration",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "benchmark_ticker": BENCHMARK_TICKER,
    "sector_universe": (
        SECTOR_UNIVERSE
        .reset_index()
        .to_dict(orient="records")
    ),

    "research_config": asdict(CONFIG),

    "timing_convention": {
        "signal_information_time": "Friday close",
        "target_calculation_time": "After Friday close",
        "nominal_rebalance_day": "Monday",
        "execution_time": (
            "Next U.S. trading session after the Friday signal"
        ),
        "holiday_rule": (
            "If Monday is not a trading day, execute on the next "
            "available U.S. trading session"
        ),
        "minimum_signal_execution_lag_sessions": 1,
        "lookahead_principle": (
            "A Friday-close signal must not influence holdings or "
            "returns before the next executable U.S. trading session."
        ),
    },

    "open_design_questions_for_block_2": [
        "Historical adjusted-price source and coverage",
        "Defensible point-in-time S&P 500 sector-weight history",
        "Final common backtest start date",
        "Treatment of XLC and XLRE pre-inception history",
        "Data caching and validation rules",
    ],
}

manifest_path = (
    DIRS["manifests"]
    / "block_1_research_configuration.json"
)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved manifest: {manifest_path}")


Saved manifest: /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/manifests/block_1_research_configuration.json


In [ ]:
# ============================================================
# BLOCK 1.9 — COMPLETION CHECK
# ============================================================

summary = {
    "Project root exists": PROJECT_ROOT.exists(),
    "Sector count": len(SECTOR_UNIVERSE),
    "Benchmark": BENCHMARK_TICKER,

    "Strategic weighting target": CONFIG.strategic_weight_method,
    "Strategic fallback": CONFIG.strategic_weight_fallback,

    "Signal frequency": CONFIG.signal_frequency,
    "Signal observation": CONFIG.signal_observation,

    "Rebalance frequency": CONFIG.rebalance_frequency,
    "Nominal rebalance day": CONFIG.rebalance_day,
    "Execution rule": CONFIG.rebalance_execution_rule,
    "Minimum lag (sessions)": (
        CONFIG.minimum_signal_execution_lag_sessions
    ),

    "Long only": CONFIG.long_only,
    "Fully invested": CONFIG.fully_invested,
    "Leverage allowed": CONFIG.leverage_allowed,

    "Active multiplier range": VALID_ACTIVE_MULTIPLIER_RANGE,
    "Parameter optimization": CONFIG.parameter_optimization_enabled,
    "Look-ahead data allowed": CONFIG.use_lookahead_data,
}

display(
    pd.Series(
        summary,
        name="Block 1 status"
    ).to_frame()
)

print("\nBLOCK 1 COMPLETE")
print(
    "Next: Block 2 — Universe, Data & "
    "Point-in-Time Strategic Weight Methodology"
)


,Block 1 status
Project root exists,True
Sector count,11
Benchmark,SPY
Strategic weighting target,SP500_SECTOR_WEIGHTS
Strategic fallback,INVERSE_VOLATILITY
Signal frequency,W-FRI
Signal observation,FRIDAY_CLOSE
Rebalance frequency,WEEKLY
Nominal rebalance day,MONDAY
Execution rule,NEXT_US_TRADING_SESSION_AFTER_SIGNAL



BLOCK 1 COMPLETE
Next: Block 2 — Universe, Data & Point-in-Time Strategic Weight Methodology
